#Configuration

In [0]:
%python
import os, json, hashlib, requests
from datetime import datetime, timezone
from pyspark.sql import functions as F
import re, json

CATALOG     = "workspace"
SCHEMA      = "sme_risk_bronze"
VOLUME      = "source_systems"
SOURCE_NAME = "asic_companies"
CKAN_ID     = "asic-companies"   # data.gov.au dataset id
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{SOURCE_NAME}"

# Find the latest company file on data.gov.au

In [0]:
%python
api = f"https://data.gov.au/data/api/3/action/package_show?id={CKAN_ID}"
pkg = requests.get(api, timeout=60).json()["result"]

csv_res = [r for r in pkg["resources"]
           if (r.get("format") or "").upper() == "CSV" and "current" in (r.get("name") or "").lower()]
assert len(csv_res) == 1, f"Expected 1 current CSV resource, found {len(csv_res)}: {[r.get('name') for r in csv_res]}"
res = csv_res[0]

source_url    = res["url"]
file_name     = source_url.split("/")[-1]
last_modified = res.get("last_modified") or res.get("metadata_modified")
snapshot_date = last_modified[:10]          # date ASIC last updated the file (YYYY-MM-DD)

print("File:          ", file_name)
print("Last modified: ", last_modified)
print("Snapshot date: ", snapshot_date)

# Download it into a dated folder

In [0]:
%python
target_dir  = f"{VOLUME_ROOT}/snapshot_date={snapshot_date}"
target_file = f"{target_dir}/{file_name}"
os.makedirs(target_dir, exist_ok=True)

if os.path.exists(target_file):
    print(f"Already downloaded, skipping: {target_file}")
else:
    tmp = target_file + ".part"
    sha = hashlib.sha256()
    with requests.get(source_url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
                sha.update(chunk)
    os.replace(tmp, target_file)     # only appears under its real name once complete

    meta = {
        "source_url": source_url,
        "file_name": file_name,
        "snapshot_date": snapshot_date,
        "source_last_modified": last_modified,
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
        "bytes": os.path.getsize(target_file),
        "sha256": sha.hexdigest(),
    }
    with open(f"{target_dir}/_metadata.json", "w") as f:
        json.dump(meta, f, indent=2)
    print(f"Saved {meta['bytes']/1e6:.1f} MB to {target_file}")

# Sanity check

In [0]:
%python
with open(target_file, encoding="utf-8", errors="replace") as f:
    header = f.readline().rstrip("\n").split("\t")
print(len(header), "columns:", header)
assert len(header) == 15, "Unexpected column count: check the file format before loading Bronze"

display(dbutils.fs.ls(target_dir))

# Create the Bronze table

In [0]:
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.asic_companies_raw"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
  company_name                    STRING,
  acn                             STRING,
  type                            STRING,
  class                           STRING,
  sub_class                       STRING,
  status                          STRING,
  date_of_registration            STRING COMMENT 'DD/MM/YYYY as supplied',
  date_of_deregistration          STRING COMMENT 'DD/MM/YYYY as supplied',
  previous_state_of_registration  STRING,
  state_registration_number       STRING,
  modified_since_last_report      STRING,
  current_name_indicator          STRING COMMENT 'Y = current-name row; null = historical name',
  abn                             STRING COMMENT '0 = no ABN',
  current_name                    STRING,
  current_name_start_date         STRING,
  _snapshot_date                  DATE      COMMENT 'Date ASIC last updated the source file',
  _source_file                    STRING    COMMENT 'Full path of the file in the volume',
  _source_url                     STRING    COMMENT 'data.gov.au download URL',
  _ingested_at                    TIMESTAMP COMMENT 'When this row was loaded'
)
COMMENT 'Raw ASIC company register, one copy per weekly snapshot. All source columns kept as strings.'
""")

In [0]:
# Table, column and volume descriptions (source: ASIC Company Dataset Help File, Appendix Item 2)

TABLE_COMMENT = (
    "Raw ASIC company register from data.gov.au (dataset asic-companies), loaded as-is, one copy per weekly snapshot. "
    "Contains all currently registered companies, companies deregistered in the past year, and historical-name rows "
    "for registered companies, so one row per company name, not per company. Source file is tab-delimited; all source columns kept as strings."
)

COLUMN_COMMENTS = {
    "company_name": "Company name as it appears on the ASIC register.",
    "acn": "Australian Company Number: 9 digits incl. leading zeros. When type = RACN this holds the ARBN (Australian Registered Body Number).",
    "type": "Company type. APTY = Australian proprietary; APUB = Australian public; FNOS = foreign company registered in Australia; "
            "RACN = registered Australian corporation (incorporated under another law). CCIV appears in the data but is not listed in the help file.",
    "class": "Liability class. LMSH = limited by shares; LMGT = limited by guarantee; LMSG = limited by shares and guarantee; "
             "NLIA = no liability (mining only); UNLM = unlimited; NONE = no Australian equivalent.",
    "sub_class": "Company sub-class, e.g. PROP = proprietary other; PNPC = proprietary non-profit; PSTC = proprietary super trustee; "
                 "HUNT = proprietary home unit; LIST = listed public; ULST = unlisted public; ULSN = unlisted public non-profit. Full list in ASIC help file.",
    "status": "Company status. REGD = registered; DRGD = deregistered; EXAD = external administration (receivership/liquidation); "
              "SOFF = strike-off action in progress; NOAC = not active; NRGD = not registered; PROV = provisional; "
              "DISS = dissolved by Act of Parliament; DIV3 = transferred via Div 3; PEND = pending (schemes); CNCL = cancelled / transferred to another law.",
    "date_of_registration": "Date the company was registered. Text, DD/MM/YYYY.",
    "date_of_deregistration": "Date the company was deregistered. Text, DD/MM/YYYY. File only includes deregistrations from the past year.",
    "previous_state_of_registration": "State where the company was originally registered, abbreviated (e.g. VIC). Only where applicable; mostly null.",
    "state_registration_number": "Number assigned by the original state registry. Format varies by state; only where applicable.",
    "modified_since_last_report": "Y if name, type, class, sub-class or status changed since the previous weekly file; otherwise null.",
    "current_name_indicator": "Y = this row shows the current company name; null = historical-name row for the same ACN.",
    "abn": "Australian Business Number. 0 = company has no ABN.",
    "current_name": "On historical-name rows (current_name_indicator null): the company current name. Null on current-name rows.",
    "current_name_start_date": "On historical-name rows: start date of the current name. Null on current-name rows.",
    "_snapshot_date": "Load metadata: date ASIC last updated the source file on data.gov.au.",
    "_source_file": "Load metadata: full path of the source file in the volume.",
    "_source_url": "Load metadata: data.gov.au download URL.",
    "_ingested_at": "Load metadata: timestamp when this row was loaded.",
}

VOLUME_COMMENT = ("Raw source files, stored unchanged. asic_companies/snapshot_date=YYYY-MM-DD/ holds each weekly ASIC "
                  "company file plus _metadata.json (source URL, ASIC last-modified date, size, SHA-256).")

def q(text):                       # escape single quotes for SQL
    return text.replace("'", "\\'")

spark.sql(f"COMMENT ON TABLE {BRONZE_TABLE} IS '{q(TABLE_COMMENT)}'")
for col, text in COLUMN_COMMENTS.items():
    spark.sql(f"ALTER TABLE {BRONZE_TABLE} ALTER COLUMN {col} COMMENT '{q(text)}'")
spark.sql(f"COMMENT ON VOLUME {CATALOG}.{SCHEMA}.{VOLUME} IS '{q(VOLUME_COMMENT)}'")

display(spark.sql(f"DESCRIBE TABLE {BRONZE_TABLE}"))

# Load function, with a row-count check

In [0]:

EXPECTED_HEADER = [
    "Company Name", "ACN", "Type", "Class", "Sub Class", "Status",
    "Date of Registration", "Date of Deregistration", "Previous State of Registration",
    "State Registration number", "Modified since last report", "Current Name Indicator",
    "ABN", "Current Name", "Current Name Start Date",
]

def to_snake(name):
    return re.sub(r"[^a-z0-9]+", "_", name.strip().lower()).strip("_")

def load_snapshot(snapshot_dir):
    meta  = json.load(open(f"{snapshot_dir}/_metadata.json"))
    path  = f"{snapshot_dir}/{meta['file_name']}"
    snap  = meta["snapshot_date"]

    df = (spark.read
          .option("header", True)
          .option("sep", "\t")
          .option("quote", "")          # file has no quoting; stops a stray " in a company name merging rows
          .option("inferSchema", False)
          .csv(path))

    # 1) Header must match what ASIC documents
    assert df.columns == EXPECTED_HEADER, f"Header changed:\n{df.columns}"

    # 2) Row count must match the raw line count (catches rows merged or split while reading)
    rows      = df.count()
    raw_lines = spark.read.text(path).filter(F.length("value") > 0).count() - 1
    assert rows == raw_lines, f"Row count {rows:,} != raw lines {raw_lines:,}"

    out = (df.toDF(*[to_snake(c) for c in df.columns])
             .withColumn("_snapshot_date", F.lit(snap).cast("date"))
             .withColumn("_source_file",   F.lit(path))
             .withColumn("_source_url",    F.lit(meta["source_url"]))
             .withColumn("_ingested_at",   F.current_timestamp()))

    # 3) Replace only this snapshot's rows, so reruns never duplicate data
    (out.write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"_snapshot_date = DATE'{snap}'")
        .saveAsTable(BRONZE_TABLE))

    print(f"Loaded snapshot {snap}: {rows:,} rows from {meta['file_name']}")
    return rows

# Load every snapshot folder not yet in the table

In [0]:
snapshot_dirs = sorted(f.path.rstrip("/") for f in dbutils.fs.ls(VOLUME_ROOT)
                       if f.name.startswith("snapshot_date="))

loaded = {r[0].isoformat() for r in
          spark.sql(f"SELECT DISTINCT _snapshot_date FROM {BRONZE_TABLE}").collect()}

for d in snapshot_dirs:
    snap = d.split("snapshot_date=")[-1]
    if snap in loaded:
        print(f"Skip {snap} (already in Bronze)")
    else:
        load_snapshot(d.replace("dbfs:", ""))

# Check the result

In [0]:
display(spark.sql(f"""
  SELECT _snapshot_date, COUNT(*) AS rows, COUNT(DISTINCT acn) AS distinct_acns,
         MIN(_ingested_at) AS ingested_at
  FROM {BRONZE_TABLE} GROUP BY _snapshot_date ORDER BY _snapshot_date
"""))
display(spark.table(BRONZE_TABLE).limit(5))
display(spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").select("version", "timestamp", "operation", "operationMetrics"))